# Interactive Orbital Dynamics Laboratory
# Laboratorio Interactivo de Dinámica Orbital

Welcome to the Interactive Orbit Laboratory. This notebook demonstrates two-body Keplerian mechanics, orbital propagation, and 3D visualization.

Bienvenido al Laboratorio de Órbitas Interactivo. Este cuaderno explora la mecánica de dos cuerpos, propagación orbital y visualización 3D.

In [ ]:
# Initialize Language Registry / Inicializar Registro de Idioma
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from orbital.i18n import Locale
from orbital.gamification import ProgressTracker

tracker = ProgressTracker()
locale = Locale("en")

print("Locale engine loaded. Default language set to English.")
print("Motor de traducciones cargado. Idioma por defecto establecido en inglés.")

## Classical Orbital Elements & Kepler's Equation
## Elementos Orbitales Clásicos y Ecuación de Kepler

The two-body equation of motion is:
$$\ddot{\vec{r}} = -\frac{\mu}{r^3}\vec{r}$$

Kepler's transcendental equation relates mean anomaly $M$ to eccentric anomaly $E$:
$$M = E - e \sin E$$

In [ ]:
# Import core packages / Importar paquetes clave
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, Markdown

from orbital.kepler import solve_kepler_equation, true_anomaly_from_eccentric
from orbital.orbits import OrbitalPropagator, create_leo_orbit, create_meo_orbit, create_geo_orbit, create_heo_orbit
from orbital.visualization import OrbitPlotter

# Record solver action
tracker.record_action("solve_kepler")
print("Kepler equation dependencies imported and first mathematical action logged.")

In [ ]:
# Visualizing Anomalies / Visualización de Anomalías
M_vals = np.linspace(0, 2 * np.pi, 500)
eccentricities = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]
colors = ['#00ff88', '#00bfff', '#ffd700', '#ff6b35', '#ff1493', '#ff0040']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Eccentric Anomaly E vs Mean Anomaly M',
        'True Anomaly nu vs Mean Anomaly M'
    ),
    horizontal_spacing=0.1
)

for e, color in zip(eccentricities, colors):
    E_vals = np.array([solve_kepler_equation(M, e) for M in M_vals])
    nu_vals = np.array([true_anomaly_from_eccentric(E, e) for E in E_vals])
    nu_vals = np.unwrap(nu_vals)
    
    fig.add_trace(
        go.Scatter(x=np.degrees(M_vals), y=np.degrees(E_vals),
                   name=f'e = {e}', line=dict(color=color, width=2.5),
                   legendgroup=f'e{e}'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=np.degrees(M_vals), y=np.degrees(nu_vals),
                   name=f'e = {e}', line=dict(color=color, width=2.5, dash='dot'),
                   legendgroup=f'e{e}', showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    template='plotly_dark',
    height=500, width=1000,
    margin=dict(t=50, b=50, l=50, r=50)
)
fig.show()

## Earth Orbits Comparison
## Comparación de Órbitas Terrestres

Comparing LEO, MEO, GEO, and HEO (Molniya) orbits side-by-side using the interactive Plotly plotter.

In [ ]:
# Define and propagate orbits / Definir e iniciar órbitas
leo = create_leo_orbit(altitude=400, inclination=51.6)
meo = create_meo_orbit(altitude=20200, inclination=55.0)
geo = create_geo_orbit(longitude=0)
heo = create_heo_orbit(perigee_alt=500, apogee_alt=39000, inclination=63.4)

plotter = OrbitPlotter(locale)
fig_compare = plotter.plot_comparison(
    [leo, meo, geo, heo],
    names=['LEO (ISS)', 'MEO (GPS)', 'GEO', 'HEO (Molniya)'],
    colors=['#00ff88', '#00bfff', '#ffd700', '#ff1493']
)

tracker.record_action("compare_four_orbits")
fig_compare.show()

## Real-time Parameter Adjustment
## Modificación de Parámetros en Tiempo Real

Use the sliders below to adjust the Keplerian elements. Changes are computed and visualized instantly.

In [ ]:
# Dynamic orbit configuration widget / Configuración dinámica de órbita
out = widgets.Output()
info_out = widgets.Output()

slider_a = widgets.FloatSlider(value=7000, min=6500, max=45000, step=250, description='a (km)')
slider_e = widgets.FloatSlider(value=0.01, min=0.0, max=0.9, step=0.01, description='e')
slider_i = widgets.FloatSlider(value=51.6, min=0.0, max=180.0, step=1.0, description='i (deg)')
slider_raan = widgets.FloatSlider(value=0, min=0, max=360, step=5, description='RAAN (deg)')
slider_argp = widgets.FloatSlider(value=0, min=0, max=360, step=5, description='argp (deg)')

lang_dropdown = widgets.Dropdown(
    options=[('English', 'en'), ('Español', 'es'), ('简体中文', 'zh')],
    value='en',
    description='Language:'
)

def update_plot(*args):
    a = slider_a.value
    e = slider_e.value
    i = slider_i.value
    raan = slider_raan.value
    argp = slider_argp.value
    lang = lang_dropdown.value
    
    loc = Locale(lang)
    
    # Check for subsurface
    if a * (1 - e) < 6371:
        with info_out:
            info_out.clear_output(wait=True)
            print(loc.t("errors.subsurface_orbit", perigee=a * (1 - e) - 6371, radius=6371))
        return
        
    prop = OrbitalPropagator(a=a, e=e, i=i, raan=raan, argp=argp, nu0=0.0)
    plt = OrbitPlotter(loc)
    fig = plt.plot_orbit(prop, color="#00ff88")
    
    with out:
        out.clear_output(wait=True)
        fig.show()
        
    with info_out:
        info_out.clear_output(wait=True)
        print(prop.summary(lang))
        
    if a != 7000 or e != 0.01 or i != 51.6:
        tracker.record_action("modify_six_elements")

slider_a.observe(update_plot, 'value')
slider_e.observe(update_plot, 'value')
slider_i.observe(update_plot, 'value')
slider_raan.observe(update_plot, 'value')
slider_argp.observe(update_plot, 'value')
lang_dropdown.observe(update_plot, 'value')

controls = widgets.VBox([lang_dropdown, slider_a, slider_e, slider_i, slider_raan, slider_argp])
display(controls, out, info_out)
update_plot()

In [ ]:
# View Unlocked Achievements / Ver Logros Desbloqueados
print(tracker.display_progress())